<a href="https://colab.research.google.com/github/abhinavnautiyalDS/Finwise-GenAI-Assistant/blob/main/finwise-genai-capstone/task-06-summarization/task_6_Summerisarion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Required packages

In [ ]:
%pip install langchain-community langchain-google-genai pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.77
    Uninstalling langchain-core-0.3.77:
      Successfully uninstalled langchain-core-0.3.77
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-gener

Importing libraries

In [ ]:
import os
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.summarize import load_summarize_chain
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from google.colab import files
from langchain.chains.summarize import load_summarize_chain
from packaging import version
import langchain


Set up GOOGLE_API_KEY

In [ ]:

# Configure Google Generative AI
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    genai.configure(api_key=GOOGLE_API_KEY)
    print("Google API Key loaded from Colab secrets.")
except Exception as e:
    print(f"Could not load API key from Colab secrets: {e}")
    print("Please ensure 'GOOGLE_API_KEY' is set in Colab secrets or as an environment variable.")

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3) # Increased temperature slightly for summarization creativity
print("Gemini Pro LLM initialized.")

Google API Key loaded from Colab secrets.
Gemini Pro LLM initialized.


PyPdf To load and parse document

In [ ]:
def load_document(file_path):
    # Determine if it's a PDF or TXT based on extension
    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
        pages = loader.load_and_split()
        print(f"Loaded {len(pages)} pages from PDF: {file_path}")
        return pages
    elif file_path.endswith('.txt'):
        loader = TextLoader(file_path)
        documents = loader.load()
        print(f"Loaded text from TXT: {file_path}")
        return documents
    else:
        print(f"Unsupported file type for: {file_path}")
        return None

# --- NEW: File Upload Section for Colab ---
print("\n--- Upload your Financial Document ---")
print("Please upload a PDF or TXT file.")

uploaded = files.upload() # This will display the upload widget in Colab

raw_documents = []
temp_file_path = None

if uploaded:
    for filename, content in uploaded.items():
        print(f"Uploaded file: {filename} ({len(content)} bytes)")

        # Save the uploaded bytes to a temporary file on disk
        temp_file_path = f"/content/{filename}"
        with open(temp_file_path, "wb") as f:
            f.write(content)

        # Load the document using the new path
        loaded_docs = load_document(temp_file_path)
        if loaded_docs:
            raw_documents.extend(loaded_docs)
            print(f"Successfully processed: {filename}")

        # Clean up temporary file
        os.remove(temp_file_path)
        print(f"Removed temporary file: {temp_file_path}")

    if not raw_documents:
        print("No documents were successfully loaded from the uploaded file(s).")
else:
    print("No file was uploaded. Please upload a document to proceed.")


--- Upload your Financial Document ---
Please upload a PDF or TXT file.


Saving Circle K India JD- Associate BI Developer (1).pdf to Circle K India JD- Associate BI Developer (1).pdf
Uploaded file: Circle K India JD- Associate BI Developer (1).pdf (158262 bytes)
Loaded 2 pages from PDF: /content/Circle K India JD- Associate BI Developer (1).pdf
Successfully processed: Circle K India JD- Associate BI Developer (1).pdf
Removed temporary file: /content/Circle K India JD- Associate BI Developer (1).pdf


Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Max characters per chunk
    chunk_overlap=100, # Overlap between chunks to maintain context
    length_function=len,
    add_start_index=True,
)

# If raw_documents is from a loader, it's already a list of Documents.
# If it's our single dummy text document, we need to split it if it's long.
if raw_documents:
    # If the text is very long, it might be a single Document. We split it.
    # Otherwise, if it's already pages from a PDF, we use those.
    if len(raw_documents) == 1 and len(raw_documents[0].page_content) > 1500: # Arbitrary threshold for "long"
        docs = text_splitter.split_documents(raw_documents)
        print(f"Document split into {len(docs)} chunks.")
    else:
        docs = raw_documents # Use as is if it's short or already split (e.g., PDF pages)
else:
    docs = []
    print("No documents to process.")

# Preview first few chunks if available
if docs:
    for i, doc in enumerate(docs[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(doc.page_content[:200] + "...") # Print first 200 chars


--- Chunk 1 ---
Couche-Tard Inc. 
 
 
 
Job Description: Associate BI Developer 
Job Description 
Job Title Associate BI Developer Direct Supervisor Senior Manager Enterprise BI 
Function Data & Analytics 
 
 
 
Alim...

--- Chunk 2 ---
Couche-Tard Inc. 
 
 
 
Qualifications 
• Bachelor’s degree in computer engineering, Computer Science, Data Analytics, or related field. 
• 1+ years of experience in BI engineering or data analytics a...


Summerisation

In [ ]:

# --- Custom Prompt Templates ---
summary_prompt_template = """Write a concise summary of the following financial document, focusing on key financial figures, strategic developments, and future outlook:

"{text}"

CONCISE SUMMARY:"""

refine_prompt_template = """Your job is to produce a final summary of the provided financial document.
We have an existing summary up to a certain point: {existing_answer}
We have the opportunity to refine the existing summary (only if needed) with some more context below:
------------
{text}
------------
Given the new context, refine the original summary to include any new key financial figures, strategic developments, or future outlook.
If the context isn't useful, return the original summary.
REFINED SUMMARY:"""

# --- Summarization Function ---
def summarize_document(documents, chain_type="map_reduce", verbose=False):
    if not documents:
        return "No documents provided for summarization."

    if chain_type == "stuff":
        # Check if total length is too large for 'stuff'
        total_length = sum(len(doc.page_content) for doc in documents)
        if total_length > 10000:  # Rough estimate based on model context size
            st.warning(f"Document length ({total_length} chars) may be too large for 'stuff'. Consider 'map_reduce' or 'refine'.")

        chain = load_summarize_chain(
            llm,
            chain_type="stuff",
            prompt=PromptTemplate(template=summary_prompt_template, input_variables=["text"]),
            verbose=verbose
        )

    elif chain_type == "map_reduce":
        map_prompt = PromptTemplate(template=summary_prompt_template, input_variables=["text"])
        combine_prompt = PromptTemplate(template=summary_prompt_template, input_variables=["text"])

        # Handle compatibility with older LangChain versions
        if version.parse(langchain.__version__) >= version.parse("0.2.0"):
            chain = load_summarize_chain(
                llm,
                chain_type="map_reduce",
                map_prompt=map_prompt,
                combine_prompt=combine_prompt,  # ✅ new name
                combine_document_variable_name="text",
                verbose=verbose
            )
        else:
            chain = load_summarize_chain(
                llm,
                chain_type="map_reduce",
                map_prompt=map_prompt,
                reduce_prompt=combine_prompt,   # ✅ backward compatible
                combine_document_variable_name="text",
                verbose=verbose
            )

    elif chain_type == "refine":
        initial_prompt = PromptTemplate(template=summary_prompt_template, input_variables=["text"])
        refine_prompt = PromptTemplate(template=refine_prompt_template, input_variables=["existing_answer", "text"])
        chain = load_summarize_chain(
            llm,
            chain_type="refine",
            question_prompt=initial_prompt,
            refine_prompt=refine_prompt,
            verbose=verbose
        )
    else:
        return "Invalid chain_type. Choose 'stuff', 'map_reduce', or 'refine'."

    print(f"\n--- Summarizing with '{chain_type}' chain ---")

    try:
        summary = chain.invoke({"input_documents": documents}, return_only_outputs=True)
        return summary['output_text']
    except Exception as e:
        return f"An error occurred during summarization: {e}"


print("\n--- Starting Summarization ---")


refine_summary = summarize_document(docs, chain_type="refine", verbose=False)
print("\n--- Refine Summary ---")
print(refine_summary)





--- Starting Summarization ---

--- Summarizing with 'map_reduce' chain ---

--- MapReduce Summary ---
This document is a job description for an Associate BI Developer at Alimentation Couche-Tard (Circle K). The company is expanding its Enterprise BI team and pursuing a cloud-first data strategy to improve reporting and insights. The role focuses on Power BI development, data governance, and report optimization to enable actionable business outcomes, with a future outlook of transforming data for long-term analytical success.
